# 📈 Demand Forecasting Dashboard
**Marketing Analytics Module 1** — Regression & neural network models with confidence intervals

Models: Linear Regression · Random Forest · MLP Neural Network  
Forecast horizon: configurable (default 4 weeks)  
Evaluation: temporal train/test split (no leakage)


In [2]:
# Load all the libraries needed for data handling, charting, and forecasting models
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, HTML
import ipywidgets as widgets
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_white"

# Assign a consistent color to each model so charts are easy to read
COLORS = {
    "Linear Regression": "#6366f1",
    "Random Forest": "#059669",
    "Neural Network (MLP)": "#f59e0b",
    "actual": "#0f172a",
    "ci": "rgba(99,102,241,0.12)",
}

# Build the styled KPI summary cards shown at the top of each SKU analysis

def metric_card(label, value, sub=""):
    return f'''<div style="flex:1;min-width:160px;background:#f8fafc;border:1px solid #e2e8f0;
    border-radius:12px;padding:1.1rem 1.4rem;">
    <div style="font-size:0.72rem;color:#64748b;text-transform:uppercase;letter-spacing:0.05em;font-weight:500;">{label}</div>
    <div style="font-size:1.45rem;font-weight:700;color:#0f172a;margin-top:0.2rem;font-family:monospace;">{value}</div>
    <div style="font-size:0.78rem;color:#64748b;margin-top:0.15rem;">{sub}</div></div>'''

def metric_row(cards):
    inner = "".join(cards)
    return HTML(f'<div style="display:flex;gap:1rem;margin:1rem 0;flex-wrap:wrap;">{inner}</div>')

print("Imports loaded")


Imports loaded


In [3]:
# Read the raw sales data and the pre-processed (feature-engineered) version
raw = pd.read_csv("data_raw.csv")
proc = pd.read_csv("data_processed.csv")
raw["week"] = pd.to_datetime(raw["week"])
proc["week"] = pd.to_datetime(proc["week"])
raw["feat_main_page"] = raw["feat_main_page"].astype(str).str.lower().eq("true").astype(int)

# Summarise each SKU: its category, color, average price, average weekly sales, and promo rate
sku_meta = raw.groupby("sku").agg(
    functionality=("functionality", "first"),
    color=("color", "first"),
    avg_price=("price", "mean"),
    avg_sales=("weekly_sales", "mean"),
    feat_rate=("feat_main_page", "mean"),
).reset_index()
sku_meta["avg_price"] = sku_meta["avg_price"].round(2)
sku_meta["avg_sales"] = sku_meta["avg_sales"].round(1)
sku_meta["feat_rate"] = (sku_meta["feat_rate"] * 100).round(1)

print(f"Raw:       {raw.shape[0]:,} rows × {raw.shape[1]} cols — {raw.sku.nunique()} SKUs × {raw.week.nunique()} weeks")
print(f"Processed: {proc.shape[0]:,} rows × {proc.shape[1]} cols")
sku_meta[["sku","functionality","color","avg_price","avg_sales","feat_rate"]].head(10)


Raw:       4,400 rows × 8 cols — 44 SKUs × 100 weeks
Processed: 4,312 rows × 48 cols


,sku,functionality,color,avg_price,avg_sales,feat_rate
0,1,06.Mobile phone accessories,black,24.01,22.2,21.0
1,2,07.Headphones,blue,64.51,8.5,8.0
2,3,07.Headphones,purple,103.58,10.6,14.0
3,4,06.Mobile phone accessories,red,19.46,8.9,50.0
4,5,06.Mobile phone accessories,red,10.94,9.4,72.0
5,6,04.Selfie sticks,blue,36.20,16.6,10.0
6,7,04.Selfie sticks,blue,11.27,85.2,6.0
7,8,03.Bluetooth speakers,black,113.01,31.2,62.0
8,9,11.Fitness trackers,black,161.80,73.6,79.0
9,10,10.VR headset,white,188.74,14.6,75.0


In [4]:
# ── Core modelling functions ──

# Pull out the input columns (price, trend, etc.) and the sales column for one SKU
def get_features_target(proc, sku_id):
    """Extract feature matrix and target for a given SKU."""
    df = proc[proc["sku"] == sku_id].sort_values("week").copy()
    feature_cols = [c for c in df.columns if c not in ["week", "sku", "weekly_sales"]]
    X = df[feature_cols].values
    y = df["weekly_sales"].values
    weeks = df["week"].values
    return X, y, weeks, feature_cols


# Train all 3 models: hold out the last N weeks as a test set to measure accuracy
def train_evaluate(X, y, weeks, test_size=8):
    """Train 3 models on temporal split, return metrics + predictions."""
    X_train, X_test = X[:-test_size], X[-test_size:]
    y_train, y_test = y[:-test_size], y[-test_size:]
    w_train, w_test = weeks[:-test_size], weeks[-test_size:]

    # Normalise features so the Neural Network can learn effectively
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Define the three forecasting models we compare
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=200, max_depth=8, min_samples_leaf=4, random_state=42, n_jobs=-1
        ),
        "Neural Network (MLP)": MLPRegressor(
            hidden_layer_sizes=(64, 32), max_iter=500, early_stopping=True,
            validation_fraction=0.15, random_state=42, learning_rate_init=0.005,
        ),
    }

    # Fit each model on training data and measure how well it predicts the test weeks
    results = {}
    for name, model in models.items():
        Xtr = X_train_s if "Neural" in name else X_train
        Xte = X_test_s if "Neural" in name else X_test
        model.fit(Xtr, y_train)
        y_pred_train = np.maximum(model.predict(Xtr), 0)
        y_pred_test = np.maximum(model.predict(Xte), 0)
        resid_std = np.std(y_train - y_pred_train)

        results[name] = {
            "model": model,
            "scaler": scaler if "Neural" in name else None,
            "y_pred_test": y_pred_test,
            "y_pred_train": y_pred_train,
            "resid_std": resid_std,
            "mae": mean_absolute_error(y_test, y_pred_test),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred_test)),
            "r2": r2_score(y_test, y_pred_test),
            "mape": np.mean(np.abs((y_test - y_pred_test) / np.maximum(y_test, 1))) * 100,
        }

    return results, y_train, y_test, w_train, w_test


# Project demand into the future using each model, with widening confidence intervals
def forecast_future(results, X, feature_cols, n_periods=4):
    """Generate future forecasts from last known features."""
    last_row = X[-1].copy().reshape(1, -1)
    forecasts = {}

    for name, res in results.items():
        preds, ci_lower, ci_upper = [], [], []
        row = last_row.copy()

        for step in range(1, n_periods + 1):
            if "trend" in feature_cols:
                trend_idx = feature_cols.index("trend")
                row[0, trend_idx] = X[-1, trend_idx] + step * (1 / len(X))

            Xf = res["scaler"].transform(row) if res["scaler"] else row
            pred = max(res["model"].predict(Xf)[0], 0)
            ci_w = res["resid_std"] * 1.96 * np.sqrt(step)
            preds.append(pred)
            ci_lower.append(max(pred - ci_w, 0))
            ci_upper.append(pred + ci_w)

        forecasts[name] = {
            "predictions": np.array(preds),
            "ci_lower": np.array(ci_lower),
            "ci_upper": np.array(ci_upper),
        }
    return forecasts

print(" Model functions defined")


 Model functions defined


In [5]:
# ── Plotting functions ──

# Draw the main forecast chart: historical sales, test-period actuals, model predictions, and future forecast
def plot_forecast(w_train, y_train, w_test, y_test, results, forecasts, best_model, sku_id, n_periods):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=w_train, y=y_train, mode="lines", name="Historical",
        line=dict(color="#cbd5e1", width=1.2),
        hovertemplate="Week: %{x|%Y-%m-%d}<br>Sales: %{y:.0f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=w_test, y=y_test, mode="lines+markers", name="Actual (test)",
        line=dict(color=COLORS["actual"], width=2.2), marker=dict(size=6),
        hovertemplate="Week: %{x|%Y-%m-%d}<br>Sales: %{y:.0f}<extra></extra>",
    ))

    last_date = pd.Timestamp(w_test[-1])
    future_weeks = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=n_periods, freq="W-MON")

    for name, res in results.items():
        is_best = name == best_model
        opacity = 1.0 if is_best else 0.4
        width = 2.5 if is_best else 1.5
        dash = None if is_best else "dot"

        fig.add_trace(go.Scatter(
            x=w_test, y=res["y_pred_test"], mode="lines",
            name=f"{name} (test)", line=dict(color=COLORS[name], width=width, dash=dash),
            opacity=opacity,
        ))

        fc = forecasts[name]
        fig.add_trace(go.Scatter(
            x=future_weeks, y=fc["predictions"], mode="lines+markers",
            name=f"{name} (forecast)", line=dict(color=COLORS[name], width=width),
            marker=dict(size=7 if is_best else 4, symbol="diamond"), opacity=opacity,
        ))

        if is_best:
            fig.add_trace(go.Scatter(
                x=list(future_weeks) + list(future_weeks[::-1]),
                y=list(fc["ci_upper"]) + list(fc["ci_lower"][::-1]),
                fill="toself", fillcolor=COLORS["ci"],
                line=dict(color="rgba(0,0,0,0)"), name="95% CI", showlegend=True,
            ))

    fig.add_shape(type="line", x0=last_date, x1=last_date, y0=0, y1=1,
                  yref="paper", line=dict(width=1.5, dash="dash", color="#94a3b8"))
    fig.add_annotation(x=last_date, y=1, yref="paper", text="Forecast →",
                       showarrow=False, font=dict(size=11, color="#64748b"),
                       xanchor="left", yanchor="top")

    fig.update_layout(
        title=f"SKU {sku_id} — Demand forecast", height=480,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, font_size=11),
        margin=dict(l=50, r=30, t=60, b=80),
        xaxis_title="Week", yaxis_title="Weekly sales (units)",
    )
    return fig


# Bar chart comparing the three models on MAE, RMSE, and R² (lower error = better, higher R² = better)
def plot_model_comparison(results):
    names = list(results.keys())
    fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE ↓", "RMSE ↓", "R² ↑"])
    for i, metric in enumerate(["mae", "rmse", "r2"]):
        vals = [results[n][metric] for n in names]
        fig.add_trace(go.Bar(
            x=names, y=vals, marker_color=[COLORS[n] for n in names], showlegend=False,
            text=[f"{v:.2f}" for v in vals], textposition="outside",
            textfont=dict(size=11, family="monospace"),
        ), row=1, col=i + 1)
    fig.update_layout(height=300, margin=dict(l=40, r=20, t=40, b=20))
    return fig


# Show prediction errors per week: positive = model underestimated, negative = overestimated
def plot_residuals(w_test, y_test, results, best_model):
    res = results[best_model]
    residuals = y_test - res["y_pred_test"]
    fig = go.Figure(go.Bar(
        x=w_test, y=residuals,
        marker_color=["#6366f1" if r >= 0 else "#ef4444" for r in residuals],
    ))
    fig.add_hline(y=0, line_color="#94a3b8", line_width=1)
    fig.update_layout(title=f"Residuals — {best_model}", height=280,
                      margin=dict(l=40, r=20, t=40, b=20),
                      xaxis_title="Week", yaxis_title="Actual − Predicted")
    return fig


# Rank which inputs (price, trend, promo, etc.) matter most for the Random Forest's predictions
def plot_feature_importance(results, feature_cols, top_n=12):
    rf = results["Random Forest"]["model"]
    imp = rf.feature_importances_
    idx = np.argsort(imp)[-top_n:]
    fig = go.Figure(go.Bar(
        x=imp[idx], y=[feature_cols[i] for i in idx],
        orientation="h", marker_color="#6366f1",
    ))
    fig.update_layout(title="Feature importance (Random Forest)", height=350,
                      margin=dict(l=180, r=20, t=40, b=20), xaxis_title="Importance")
    return fig

print("Plotting functions defined")


Plotting functions defined


## Interactive SKU analysis
Select a SKU and configure the forecast below. 

For each selected SKU, it filters the processed dataset to that SKU's 98 weekly observations, then splits temporally — the last N weeks (default 8) become the test set, everything before is training. Three models (Linear Regression on raw features, Random Forest on raw features, MLP on StandardScaler-transformed features) are each fit on the training portion and evaluated against the held-out test weeks. The best model is auto-selected by lowest test MAE.

For the forecast, it takes the last known feature row and projects it forward by incrementing the trend variable by 1/n_obs per step (simulating linear trend continuation), then predicts with each model. Confidence intervals come from the training residuals: σ_residual × 1.96 × √(step), so uncertainty widens with each forecast week — reflecting that near-term predictions are more reliable than distant ones.

The KPI cards, forecast chart, comparison bars, residual plot, and feature importance all regenerate on each button click via ipywidgets.Output — everything stays in-notebook with no external server.

Reference: Cohen, M. C., Gras, P.E., Pentecoste, A., & Zhang, R. (2022). Demand Prediction in Retail - A Practical Guide to Leverage Data and Predictive Analytics. Springer Series in Supply Chain Management 14, 1-155.

In [ ]:
# ── Interactive controls: pick a SKU, set forecast horizon, and click Run ──
sku_list = sorted(proc["sku"].unique())
sku_labels = {s: f"SKU {s} — {sku_meta[sku_meta.sku==s].functionality.values[0]}" for s in sku_list}

w_sku = widgets.Dropdown(options=[(sku_labels[s], s) for s in sku_list], value=1, description="SKU:")
w_horizon = widgets.IntSlider(value=4, min=2, max=12, step=1, description="Weeks:")
w_test = widgets.IntSlider(value=8, min=4, max=16, step=1, description="Test size:")
w_run = widgets.Button(description="Run forecast", button_style="success", icon="play")
output = widgets.Output()

# When the button is clicked: train models, forecast, and display all charts and tables
def run_forecast(_):
    output.clear_output(wait=True)
    with output:
        sku_id = w_sku.value
        n_periods = w_horizon.value
        test_size = w_test.value

        X, y, weeks, feature_cols = get_features_target(proc, sku_id)
        if len(X) < test_size + 10:
            print(f"⚠ Insufficient data for SKU {sku_id}")
            return

        results, y_train, y_test, w_train, w_test_arr = train_evaluate(X, y, weeks, test_size)
        forecasts = forecast_future(results, X, feature_cols, n_periods)
        
        # Pick the model with the lowest average error on the test weeks
        best_model = min(results, key=lambda k: results[k]["mae"])
        best = results[best_model]
        fc_best = forecasts[best_model]

        # Show SKU summary: category, color, avg price, avg sales, promo rate
        sm = sku_meta[sku_meta.sku == sku_id].iloc[0]
        display(HTML(f"<h3>SKU {sku_id} — {sm.functionality}</h3>"
                     f"<p style='color:#64748b;'>Color: {sm.color} · Avg price: ${sm.avg_price} · "
                     f"Avg sales: {sm.avg_sales}/wk · Promo rate: {sm.feat_rate}%</p>"))

        # Display key performance indicators: best model, error, R², and forecast vs recent trend
        avg_hist = np.mean(y[-12:])
        avg_fc = np.mean(fc_best["predictions"])
        delta = ((avg_fc - avg_hist) / max(avg_hist, 1)) * 100
        delta_sym = "▲" if delta >= 0 else "▼"
        display(metric_row([
            metric_card("Best model", best_model.split("(")[0].strip(), "Lowest MAE on test"),
            metric_card("Test MAE", f"{best['mae']:.1f}", "units/week"),
            metric_card("Test R²", f"{best['r2']:.3f}", "Good ✓" if best['r2'] > 0.5 else "Moderate"),
            metric_card(f"Avg forecast ({n_periods}w)", f"{avg_fc:.0f}", f"{delta_sym} {abs(delta):.1f}% vs recent"),
        ]))

        # Plot the timeline: past sales, test actuals, model predictions, and future forecast
        fig = plot_forecast(w_train, y_train, w_test_arr, y_test, results, forecasts, best_model, sku_id, n_periods)
        fig.show()

        # Build a week-by-week forecast table with predictions from each model and confidence range
        last_date = pd.Timestamp(w_test_arr[-1])
        rows = []
        for step in range(n_periods):
            wk = last_date + pd.Timedelta(weeks=step + 1)
            row = {"Week": wk.strftime("%Y-%m-%d")}
            for name in results:
                row[name] = f"{forecasts[name]['predictions'][step]:.0f}"
            row["95% CI (best)"] = f"[{fc_best['ci_lower'][step]:.0f}, {fc_best['ci_upper'][step]:.0f}]"
            rows.append(row)
        display(HTML("<h4>Forecast detail</h4>"))
        display(pd.DataFrame(rows))

        # Side-by-side bar chart: which model had the lowest error and best fit?
        display(HTML("<h4>Model comparison</h4>"))
        comp = plot_model_comparison(results)
        comp.show()

        # Metrics table
        rows_m = []
        for name, res in results.items():
            rows_m.append({"Model": name, "MAE": f"{res['mae']:.2f}", "RMSE": f"{res['rmse']:.2f}",
                          "MAPE %": f"{res['mape']:.1f}", "R²": f"{res['r2']:.3f}",
                          "Best": "★" if name == best_model else ""})
        display(pd.DataFrame(rows_m))

        # Residual plot (where did the model over/under-predict?) and feature importance ranking
        display(HTML("<h4>Diagnostics</h4>"))
        plot_residuals(w_test_arr, y_test, results, best_model).show()
        plot_feature_importance(results, feature_cols).show()

w_run.on_click(run_forecast)
display(widgets.HBox([w_sku, w_horizon, w_test, w_run]))
display(output)


Output()

## All-SKU forecast heatmap
Quick Random Forest forecast across all 44 SKUs.

In [7]:
# Forecast the next 4 weeks for every SKU at once using Random Forest
n_periods = 4
all_preds = {}

for sku_id in sorted(proc["sku"].unique()):
    X, y, weeks, feature_cols = get_features_target(proc, sku_id)
    if len(X) < 20:
        continue
    model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(X, y)
    last_row = X[-1].copy().reshape(1, -1)
    preds = []
    for step in range(1, n_periods + 1):
        if "trend" in feature_cols:
            last_row[0, feature_cols.index("trend")] = X[-1, feature_cols.index("trend")] + step / len(X)
        preds.append(max(model.predict(last_row)[0], 0))
    all_preds[sku_id] = preds

# Display results as a heatmap: darker blue = higher predicted demand
sku_labels = [f"SKU {s}" for s in all_preds.keys()]
z = np.array(list(all_preds.values()))
last_date = pd.Timestamp(proc["week"].max())
week_labels = [(last_date + pd.Timedelta(weeks=i)).strftime("%d %b") for i in range(1, n_periods + 1)]

fig = go.Figure(data=go.Heatmap(
    z=z, x=week_labels, y=sku_labels, colorscale="Blues",
    text=np.round(z, 0), texttemplate="%{text:.0f}", textfont=dict(size=9),
    hovertemplate="SKU: %{y}<br>Week: %{x}<br>Demand: %{z:.0f}<extra></extra>",
))
fig.update_layout(title="Forecasted demand — all SKUs (Random Forest)",
                  height=max(500, len(sku_labels) * 22),
                  margin=dict(l=70, r=20, t=50, b=30), yaxis=dict(dtick=1))
fig.show()
print(f"Forecasted {len(all_preds)} SKUs × {n_periods} weeks")


Forecasted 44 SKUs × 4 weeks


## Methodology

**Data:** 44 SKUs × 98 weeks from the processed dataset with lagged prices, trend, month dummies, and one-hot categoricals.

**Train/test split:** Last N weeks held out (temporal split — no leakage).

**Models:**
- **Linear Regression** — OLS baseline on raw features
- **Random Forest** — 200 trees, max_depth=8, min_samples_leaf=4
- **Neural Network (MLP)** — 2 hidden layers (64→32), scaled inputs, early stopping

**Confidence intervals:** Training residual σ × 1.96 × √(step) — widens with forecast horizon. 95% level.

**Best model selection:** Lowest MAE on the held-out test set.
